# Milestone - Unit 10: approximate nearest neighbours and hybrid retrieval -- every dense path so far (the Unit-7 semantic cosine, the Unit-8 two-tower, the Unit-9 feature tower) answers a query by scoring *all* 2,000 book vectors by brute force and sorting -- linear in the catalog, fine here and hopeless at a million. This milestone builds a FAISS **HNSW** graph index over the two-tower's item vectors and measures the **recall/speed trade**: at the pinned `efSearch=64` it recovers the exact top-10 at recall ~0.999 and reproduces the two-tower's hit@10 *exactly* (0.340) -- ANN is a **speed** technique that *preserves* accuracy, with no speedup at 2,000 items but an asymptotic one (a 20,000-book synthetic index runs several times faster, ~3-5x, machine-dependent -- widening toward an order of magnitude at larger catalogs). It then **fuses** the sparse Unit-3 BM25 path with the dense two-tower into a `HybridRetrievalPath`: a tuned, dense-heavy fusion (`pool=50, weight=0.7`) edges the two-tower on both hit@10 (0.362) and coverage (0.192), beating the Unit-6 blend on hit (0.362 > 0.306) while losing to it on coverage (0.192 < 0.333) -- a modest, tuning-dependent complement, not a new best.

## 1. Load the catalog, interactions, and cold-reader partition

The same generated substrate Units 1-9 ran on -- the catalog, the implicit-feedback interaction log (train / val / test splits) and the committed cold-reader partition every scoreboard excludes. This unit **indexes and fuses the paths that already exist**, so there is no new data and no new model: the Unit-8 two-tower is the dense source and the Unit-3 BM25 path is the sparse one. We never touch `split="test"` -- that holdout is sealed for Checkpoint B. `bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been built.

In [ ]:
import json
from time import perf_counter

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    AnnRetrievalPath,
    HnswIndex,
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PathRegistry,
    PopularityRetrievalPath,
    TwoTowerRetrievalPath,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

K = 10
SEED = 0
M = 32                  # HNSW graph degree
EF_CONSTRUCTION = 200   # build-time breadth
EF_SEARCH = 64          # query-time recall/speed knob (pinned)

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold_partition = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold_partition["cold_readers"]

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")

{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "cold_readers_excluded": len(cold_readers),
}

## 2. Fit the two-tower and build the HNSW index (one seeded fit)

The dense source is the Unit-8 ID-only two-tower (seed 0), the book's strongest warm path -- one real torch fit (~15 s), every RNG fixed. Its `artifact()` exposes the learned reader and item matrices in one 32-dim inner-product space; the item matrix is exactly what every dense path scans by brute force. We hand that matrix to `HnswIndex`, which builds a FAISS `IndexHNSWFlat(dim, M, METRIC_INNER_PRODUCT)` -- **inner product, not the FAISS default L2** -- so the graph ranks the SAME `reader . item` dot product the exact scan uses (an L2 index would rank differently and recall-vs-exact would be meaningless). The build is single-threaded (`faiss.omp_set_num_threads(1)`, saved and restored) and therefore reproducible.

In [ ]:
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)  # ~15 s, one torch fit

art = two_tower.artifact()
item_matrix = np.array(art["item_embeddings"], dtype=np.float32)
reader_matrix = np.array(art["reader_embeddings"], dtype=np.float32)
item_ids = [int(i) for i in art["item_ids"]]

# The HNSW graph index over the two-tower's item vectors. METRIC_INNER_PRODUCT ranks the same dot
# product the exact scan uses; the build is single-threaded and deterministic.
build_start = perf_counter()
index = HnswIndex(item_matrix, item_ids, m=M, ef_construction=EF_CONSTRUCTION)
build_seconds = perf_counter() - build_start

assert index.size == len(catalog_ids) == 2000   # one graph node per book
assert item_matrix.shape == (2000, 32)           # 2,000 books in the 32-dim two-tower space

{
    "readers": reader_matrix.shape[0],
    "books_indexed": index.size,
    "embedding_dim": item_matrix.shape[1],
    "hnsw_build_seconds": round(build_seconds, 3),
}

## 3. Recall@10 vs exact, across an `efSearch` sweep

The exact brute-force top-10 is an `argsort` over the full `reader . item` score matrix -- what every dense path already does. ANN is **approximate**: HNSW visits a few graph neighbours instead of all 2,000 books, so we verify it by **recall@10 against that exact top-10**, never by exact neighbour identity (design 011 §7). `efSearch` is the recall/speed knob -- higher explores more of the graph (higher recall, slower). Recall climbs from ~0.98 at the cheapest search to ~1.0, and at the pinned `efSearch=64` it is ~0.999. The latency column is the batched query over the whole ~540-reader cohort, in milliseconds -- but read it honestly: **at 2,000 books there is no speedup to be had** (the exact scan is already microseconds), so this is an absolute ceiling, not a win; the win is asymptotic (next section). Two independent single-thread builds agree to **rank overlap 1.0** with allclose scores (determinism, design 011 §184 -- rank overlap + tolerance, not exact-id identity).

In [ ]:
# Exact top-10 by brute force: the reference every dense path already computes.
exact_scores = reader_matrix @ item_matrix.T
exact_top10 = np.argsort(-exact_scores, axis=1)[:, :K]


def recall_at_10(ef):
    """Mean per-reader overlap between the ANN top-10 and the exact top-10 (both in row space)."""
    labels, _ = index.search(reader_matrix, K, efSearch=ef)
    overlaps = []
    for i in range(reader_matrix.shape[0]):
        ann_rows = {int(x) for x in labels[i] if x >= 0}
        overlaps.append(len(ann_rows & {int(x) for x in exact_top10[i]}) / K)
    return float(np.mean(overlaps))


sweep_rows = []
for ef in (8, 16, 32, 64, 128):
    query_start = perf_counter()
    index.search(reader_matrix, K, efSearch=ef)
    cohort_ms = (perf_counter() - query_start) * 1000.0  # timing-dependent: a ceiling, not a speedup
    sweep_rows.append({
        "efSearch": ef,
        "recall@10": round(recall_at_10(ef), 4),
        "cohort_query_ms": round(cohort_ms, 2),
    })

recall_by_ef = {entry["efSearch"]: entry["recall@10"] for entry in sweep_rows}

# Determinism (design 011 §184): a second single-thread build gives the SAME neighbours -- gated on
# top-k RANK OVERLAP (1.0 here) + allclose scores, NOT exact neighbour-id identity and NOT exact-float
# against the brute force (ANN is approximate by construction).
second_index = HnswIndex(item_matrix, item_ids, m=M, ef_construction=EF_CONSTRUCTION)
labels_a, scores_a = index.search(reader_matrix, K, efSearch=EF_SEARCH)
labels_b, scores_b = second_index.search(reader_matrix, K, efSearch=EF_SEARCH)
build_overlap = float(np.mean([
    len({int(x) for x in labels_a[i] if x >= 0} & {int(x) for x in labels_b[i] if x >= 0}) / K
    for i in range(reader_matrix.shape[0])
]))
assert build_overlap >= 0.99
assert np.allclose(scores_a, scores_b)

# Recall rises from the cheapest to the broadest search (endpoints, NOT strict per-step -- it
# plateaus near 1.0), and the pinned efSearch=64 clears the 0.98 gate.
assert recall_by_ef[8] <= recall_by_ef[128] + 1e-9
assert recall_by_ef[8] >= 0.95
assert recall_by_ef[64] >= 0.98

pd.DataFrame(sweep_rows)

## 4. Where ANN pays off: a 20,000-book synthetic index

At 2,000 books HNSW and the exact scan are both microseconds -- the win is **asymptotic**. To show it without a new model, we jitter 10 slightly-perturbed copies of each item vector into a **20,000**-book synthetic catalog (not 200,000, which would blow the CI build budget) and index that. Recall@10 stays high (~0.99), the build takes a couple of seconds, and now the approximate search is meaningfully faster than the exact matrix scan -- several times (~3-5x) at 20k, machine-dependent (widening toward an order of magnitude at larger catalogs). The measured wall-clock ratio is *recorded, not gated* (it varies run to run); the cell gates on deterministic facts only. Teaching point: at a *fixed* `efSearch` recall falls as the catalog grows, so the knob must scale with the catalog.

In [ ]:
rng = np.random.default_rng(SEED)
copies = 10
base = np.repeat(item_matrix, copies, axis=0)
jitter = rng.normal(0.0, 0.01, base.shape).astype(np.float32)
big_matrix = (base + jitter).astype(np.float32)
big_ids = list(range(big_matrix.shape[0]))

big_build_start = perf_counter()
big_index = HnswIndex(big_matrix, big_ids, m=M, ef_construction=EF_CONSTRUCTION)
big_build_seconds = perf_counter() - big_build_start

# Recall@10 vs the exact scan over the 20k catalog.
big_exact_top10 = np.argsort(-(reader_matrix @ big_matrix.T), axis=1)[:, :K]
big_labels, _ = big_index.search(reader_matrix, K, efSearch=EF_SEARCH)
big_overlaps = []
for i in range(reader_matrix.shape[0]):
    ann_rows = {int(x) for x in big_labels[i] if x >= 0}
    big_overlaps.append(len(ann_rows & {int(x) for x in big_exact_top10[i]}) / K)
big_recall = float(np.mean(big_overlaps))

# Latency: the exact scan is matmul + argpartition over ALL 20k books; ANN visits a few neighbours.
exact_start = perf_counter()
scan = reader_matrix @ big_matrix.T
np.argpartition(-scan, K, axis=1)[:, :K]
exact_ms = (perf_counter() - exact_start) * 1000.0
ann_start = perf_counter()
big_index.search(reader_matrix, K, efSearch=EF_SEARCH)
ann_ms = (perf_counter() - ann_start) * 1000.0
speedup = exact_ms / ann_ms  # RECORDED, not gated: a wall-clock ratio is machine-dependent

# Gate on DETERMINISTIC facts only (exactly as the lesson's latency cell does): the 20k index is
# built, recall stays high, and the whole-cohort ANN query clears a generous absolute ceiling. The
# speedup is PRINTED as prose below, never asserted -- a wall-clock ratio is not a stable gate.
assert big_index.size == 20000
assert big_recall >= 0.95   # recall stays high at 20k (measured ~0.99)
assert ann_ms < 1000.0      # the whole-cohort ANN query clears a generous < 1 s ceiling

print(f"measured 20k speedup (RECORDED, not gated; machine-dependent): exact {exact_ms:.1f} ms vs "
      f"ANN {ann_ms:.1f} ms = ~{speedup:.1f}x -- several times faster at 20k, widening toward an "
      f"order of magnitude at larger catalogs.")

{
    "books_indexed": big_index.size,
    "build_seconds": round(big_build_seconds, 2),
    "recall@10": round(big_recall, 4),
    "exact_scan_ms": round(exact_ms, 2),
    "ann_search_ms": round(ann_ms, 2),
    "approx_speedup": round(speedup, 1),
}

## 5. The `val` scoreboard: ANN preserves accuracy; hybrid is a small, two-edged lift

Now the two paths this unit ships, both on the frozen `val` split (500 eligible readers, the 60 cold readers excluded -- never `test`). We register the `AnnRetrievalPath` over the two-tower and the `HybridRetrievalPath(pool=50, weight=0.7)` fusing Unit-3 BM25 with the two-tower, then read two things honestly:

- **ANN preserves accuracy.** Its hit@10 is *identical* to the exact two-tower (0.340) -- the ~0.999 recall means the handful of reordered neighbours never cost a hit. ANN buys speed at scale, nothing on accuracy.
- **Hybrid edges the two-tower on both axes, but barely.** `pool=50, weight=0.7` lifts hit@10 to 0.362 and coverage to 0.192 -- BM25 recovers a few lexically-matched books the dense tower ranks low. The gain is small and weight-sensitive: equal weights *hurt* and reciprocal-rank fusion *loses* to the two-tower alone (both recorded below). Fusion is not a free win.

In [ ]:
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
ann = AnnRetrievalPath(two_tower, efSearch=EF_SEARCH, m=M, ef_construction=EF_CONSTRUCTION).fit(
    rows, catalog=catalog_ids
)
hybrid = HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50, method="weighted").fit(rows)

registry = PathRegistry()
for path in (lexical, two_tower, ann, hybrid):
    registry.register(path)
assert ann.artifact_name() == "ann-v1"
assert hybrid.artifact_name() == "hybrid-v1"


def board(path, pool=K):
    """hit@10 + catalog coverage via the single-path blended scoreboard (reproduces the path's hit@k)."""
    result = run_blended_scoreboard(
        {path.name: path}, interactions_path, catalog_ids=catalog_ids, k=K, pool=pool,
        cold_readers=cold_readers,
    )
    return result.hit_rate_at_k, result.catalog_coverage


scoreboard_rows = []
for name, path in (
    ("bm25 (sparse)", lexical),
    ("two-tower (dense)", two_tower),
    ("ann (approx two-tower)", ann),
    ("hybrid pool50 w0.7", hybrid),
):
    hit, coverage = board(path)
    scoreboard_rows.append({"path": name, "hit@10": round(hit, 4), "coverage": round(coverage, 4)})

# ANN is a SPEED technique, not an accuracy one: its val hit@10 equals the exact two-tower (delta 0).
tt_hit = run_validation_scoreboard(
    two_tower, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
).hit_rate_at_k
ann_hit = run_validation_scoreboard(
    ann, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
).hit_rate_at_k
assert round(ann_hit, 3) == round(tt_hit, 3)   # identical hit@10 -- ANN preserves accuracy
assert abs(tt_hit - 0.340) < 0.001

hybrid_hit, hybrid_cov = board(hybrid)
tt_cov = board(two_tower)[1]

# Equal weights HURT and standard RRF loses to the two-tower alone -- fusion is NOT a free win.
equal_hit, _ = board(HybridRetrievalPath(lexical, two_tower, weight=0.5, pool=50).fit(rows))
rrf_hit, _ = board(HybridRetrievalPath(lexical, two_tower, weight=0.5, pool=50, method="rrf").fit(rows))
assert equal_hit < tt_hit
assert rrf_hit < tt_hit

# The pinned dense-heavy hybrid edges the two-tower on BOTH axes (small, weight-sensitive,
# measured-safe -- directional, NOT a strict hybrid > two-tower on every reader).
assert hybrid_hit >= tt_hit - 0.01
assert hybrid_cov > tt_cov
assert abs(hybrid_hit - 0.362) < 0.003 and abs(hybrid_cov - 0.192) < 0.003

print(f"fusion is tuning-dependent, not free: equal-weights hybrid hit@10 {equal_hit:.3f} and "
      f"RRF hit@10 {rrf_hit:.3f} both LOSE to the two-tower's {tt_hit:.3f}.")
pd.DataFrame(scoreboard_rows)

## 6. Hybrid vs the Unit-6 blend -- relevance up, catalog narrower

Finally, the honest head-to-head. The Unit-6 pinned blend Checkpoint A froze -- `{popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}` at per-path pool 30 -- is a *broad* recommender: four complementary signals, hit@10 0.306 and the book's widest catalog coverage (0.333). The sparse+dense hybrid is a *sharp* one: hit@10 0.362 but coverage only 0.192. Rebuilding the Unit-6 blend needs one more real fit (matrix factorization, ~15 s numpy gradient descent).

In [ ]:
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)  # ~15 s, numpy gradient descent

PINNED_U6_WEIGHTS = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}
u6_paths = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}
u6 = run_blended_scoreboard(
    u6_paths, interactions_path, catalog_ids=catalog_ids, k=K, pool=30,
    weights=PINNED_U6_WEIGHTS, cold_readers=cold_readers,
)

# Measured: U6 blend hit 0.306 / coverage 0.333; hybrid hit 0.362 / coverage 0.192.
assert abs(u6.hit_rate_at_k - 0.306) < 0.003
assert abs(u6.catalog_coverage - 0.333) < 0.003
# Hybrid WINS on hit@10 but LOSES on coverage -- frame BOTH axes; neither recommender dominates.
assert hybrid_hit > u6.hit_rate_at_k        # 0.362 > 0.306
assert hybrid_cov < u6.catalog_coverage     # 0.192 < 0.333

pd.DataFrame([
    {"recommender": "unit-6 blend (4-way, broad)",
     "hit@10": round(u6.hit_rate_at_k, 4), "coverage": round(u6.catalog_coverage, 4)},
    {"recommender": "unit-10 hybrid (sparse+dense, sharp)",
     "hit@10": round(hybrid_hit, 4), "coverage": round(hybrid_cov, 4)},
])

**The honest headline.** Approximate nearest neighbours index the two-tower's item vectors in a FAISS HNSW graph and recover the exact top-10 at recall ~0.999, so hit@10 is *unchanged* (0.340) -- ANN is a **speed** technique that *preserves* accuracy. At 2,000 books there is no speedup; at 20,000 the approximate search is several times faster (~3-5x, machine-dependent, widening toward an order of magnitude at larger catalogs), and the `efSearch` knob trades recall for latency. The sparse+dense **hybrid** (`pool=50, weight=0.7`) is a modest, tuning-dependent lift -- it edges the two-tower on both hit@10 (0.362) and coverage (0.192) and beats the Unit-6 blend on hit (0.362 > 0.306) while losing to it on coverage (0.192 < 0.333). Equal weights hurt and RRF loses: fusion is not a free win.

**The bridge.** Retrieval is now both accurate and (at scale) fast, but it still only *retrieves*. Unit 11 adds a learned **reranker** over the retrieved pool, and Unit 12 brings in **sequence** models that read the order of a reader's history.